# E-commerce: средний чек по географии и прогноз повторных покупок

Данные: публичный датасет [Online Retail](https://www.kaggle.com/datasets/carrie1/ecommerce-data) (транзакции британского онлайн-магазина за 2010–2011). Файл `data.csv` в репозиторий не входит.

Два вопроса:
1. Отличается ли средний чек у клиентов из Великобритании и из других стран?
2. Можно ли по истории покупок предсказать, вернётся ли клиент за повторной покупкой?

Очистка: убираю строки без `CustomerID` (без него нельзя агрегировать по клиенту) и строки с неположительными `Quantity` / `UnitPrice` — это возвраты и отмены.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score

import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('data.csv', encoding='unicode_escape')

df = df.dropna(subset=['CustomerID']) 
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

df['TotalSpend'] = df['Quantity'] * df['UnitPrice']
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print(f"Строк после очистки: {len(df)}")

## Средний чек: UK против остальных стран

Агрегирую до уровня клиента и делю на две независимые выборки. Размеры и дисперсии групп сильно отличаются, поэтому использую t-критерий Уэлча (без предположения о равных дисперсиях).

In [ ]:

customer_aov = df.groupby(['CustomerID', 'Country'])['TotalSpend'].mean().reset_index()
customer_aov.columns = ['CustomerID', 'Country', 'AOV']


customer_aov['Is_UK'] = customer_aov['Country'] == 'United Kingdom'
uk_aov = customer_aov[customer_aov['Is_UK']]['AOV']
intl_aov = customer_aov[~customer_aov['Is_UK']]['AOV']

t_stat, p_val = stats.ttest_ind(uk_aov, intl_aov, equal_var=False)

print(f"Средний чек (UK): £{uk_aov.mean():.2f}")
print(f"Средний чек (International): £{intl_aov.mean():.2f}")
print(f"T-statistic: {t_stat:.2f}")
print(f"P-value: {p_val:.5f}")

Распределения сильно скошены вправо. Для наглядности обрезаю значения выше 1000 £ (только на графике, тест считался по всем данным).

In [ ]:
plt.figure(figsize=(10, 6))
sns.set_style("whitegrid")

# обрезаю выбросы только для графика
sns.kdeplot(uk_aov[uk_aov < 1000], label='UK Customers', fill=True, color='blue', alpha=0.5)
sns.kdeplot(intl_aov[intl_aov < 1000], label='International Customers', fill=True, color='orange', alpha=0.5)

plt.title('Distribution of Average Order Value (AOV): UK vs International', fontsize=14)
plt.xlabel('Average Order Value (£)', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.legend()

plt.savefig('aov_distribution_kde.png', dpi=300, bbox_inches='tight')
plt.show()

## Прогноз повторной покупки

Признаки RFM на уровне клиента: Recency (дней с последней покупки), Frequency (число заказов), Monetary (сумма). Целевая переменная: клиент сделал больше одного заказа.

Frequency напрямую определяет целевую переменную, поэтому в модель её не беру. Monetary тоже исключаю, потому что он растёт с числом заказов. Признаки: Recency и средний чек на заказ (AOV). Модель: случайный лес, метрика ROC-AUC.

In [ ]:

snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days, # Recency
    'InvoiceNo': 'nunique',                                  # Frequency
    'TotalSpend': 'sum'                                      # Monetary
}).reset_index()

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

rfm['AOV'] = rfm['Monetary'] / rfm['Frequency']

rfm['Retained'] = (rfm['Frequency'] > 1).astype(int)

X = rfm[['Recency', 'AOV']]
y = rfm['Retained']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5)
rf_model.fit(X_train, y_train)

y_probs = rf_model.predict_proba(X_test)[:, 1]
y_pred = rf_model.predict(X_test)

auc_score = roc_auc_score(y_test, y_probs)
accuracy = accuracy_score(y_test, y_pred)

print(f"Model Accuracy: {accuracy:.2f}")
print(f"AUC-ROC Score: {auc_score:.2f}")

## Выводы

- Средний чек у клиентов вне UK статистически значимо выше (p < 0,001), у них заметнее длинный правый хвост крупных заказов. Имеет смысл проверить, окупается ли привлечение международных клиентов, с учётом стоимости доставки.
- Модель даёт ROC-AUC около 0,79: по истории можно ранжировать клиентов по вероятности вернуться и отправлять удерживающие предложения тем, у кого она ниже.

## Ограничения

- Recency считается по всему периоду, включая саму повторную покупку, поэтому частично «подсматривает» в целевую переменную. Корректнее разделить период по времени: признаки до даты отсечения, целевая переменная после неё. Тогда AUC, скорее всего, будет ниже.
- В тесте средний чек считается как среднее по строкам транзакций, а не по заказам. Правильнее сначала сложить строки внутри `InvoiceNo`.
- Распределения чеков тяжелохвостые. Для проверки устойчивости вывода стоит добавить критерий Манна–Уитни или бутстреп-интервал для разницы средних.

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_probs)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Random Forest (AUC = {auc_score:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Guessing')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve: Predicting Customer Retention', fontsize=14)
plt.legend(loc="lower right")

plt.savefig('rf_roc_curve_fixed.png', dpi=300, bbox_inches='tight')
plt.show()